# Dioptra-DINO: End-to-End Architectural Ablation Suite
### Retraining Component Controls from Scratch (Epoch 0 to 40) on Dual NVIDIA T4 GPUs

This notebook provides the complete reproducible environment to retrain all headline ablation variants of **Dioptra-DINO** from scratch:

| Ablation Variant | Description | Architecture / Loss Changes | Target Parameters |
| :--- | :--- | :--- | :---: |
| **1. `no-ara`** | Without Angular Residual Attention | `enable_ara=False` (no geometric attention bias) | $26.33$\,M |
| **2. `center-ray`** | Center-Ray Only PE | `ray_mode="center_ray"` (1 ray per patch, 36 dims) | $27.49$\,M |
| **3. `no-ray`** | Canonical 2D ViT + DPT | `enable_trivision=False` (no ray unprojection) | $26.10$\,M |
| **4. `no-vnl`** | Without 3D Virtual Normal Loss | `weight_normal=0.0` (pure SiLog + Scale + Edge) | $27.51$\,M |
| **5. `no-scale-loss`** | Without Global Scale Loss | `weight_scale=0.0` (unsupervised scale) | $27.51$\,M |
| **6. `no-dynamic-crop`** | Without Optical Zoom Augmentation | Fixed camera calibration matrix $K$ | $27.51$\,M |


In [ ]:
# [1] Hardware & Runtime Setup (Dual T4 GPUs)
import os, sys, shutil, glob
import torch

print("=" * 75)
print("DIOPTRA-DINO ABLATION SUITE: HARDWARE VERIFICATION")
print("=" * 75)

if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    gpu_name = torch.cuda.get_device_name(0)
    print(f"Detected {gpu_count}x GPU(s): {gpu_name}")
    print(f"Total VRAM: {gpu_count * 16} GB (Dual T4 setup)")
else:
    print("WARNING: No CUDA device found. Please select 'GPU T4 x2' in Kaggle settings.")

# Ensure repository code is in sys.path
repo_candidates = ['/kaggle/working', '/kaggle/working/dioptra_repo', '.']
for r in repo_candidates:
    if os.path.exists(r) and r not in sys.path:
        sys.path.insert(0, r)

print("Python runtime and CUDA environment verified!")


In [ ]:
# [2] Verify TartanAir Warehouse Stereo Suite Dataset
from dioptra_dino import resolve_dataset_root

try:
    data_root = resolve_dataset_root("auto")
    print(f"Active TartanAir dataset root: {data_root}")
except Exception as e:
    print(f"Dataset discovery note: {e}")
    print("Please ensure 'tartanair-warehouse-stereo-suite' (by yumnamharryson) is attached as Input.")


--- 
### Select Which Ablation(s) to Run
Run any of the cells below. Each variant will train independently from Epoch 0 to 40, logging progress and saving checkpoints to `outputs_ablations/<ablation_name>/`.


In [ ]:
# [Ablation 1] Retrain WITHOUT Angular Residual Attention (enable_ara=False)
# Tests: Edge sharpening and normal regularization across structural boundaries
!python scripts/train_dino_ablation.py --ablation no-ara --epochs 40 --batch-size 8 --accum-steps 4


In [ ]:
# [Ablation 2] Retrain with CENTER-RAY ONLY (ray_mode='center_ray')
# Tests: Frustum divergence aperture vs single optical coordinate (36 vs 108 dims)
!python scripts/train_dino_ablation.py --ablation center-ray --epochs 40 --batch-size 8 --accum-steps 4


In [ ]:
# [Ablation 3] Retrain WITHOUT Ray Positional Modulation (Canonical 2D ViT + DPT)
# Tests: Fundamental metric scale grounding provided by camera ray unprojection
!python scripts/train_dino_ablation.py --ablation no-ray --epochs 40 --batch-size 8 --accum-steps 4


In [ ]:
# [Ablation 4] Retrain WITHOUT 3D Virtual Normal Loss (weight_normal=0.0)
# Tests: Impact of 3D planar surface regularization on floor/wall orientation
!python scripts/train_dino_ablation.py --ablation no-vnl --epochs 40 --batch-size 8 --accum-steps 4


In [ ]:
# [Ablation 5] Retrain WITHOUT Global Log-Median Scale Loss (weight_scale=0.0)
# Tests: Scale drift stability without explicit metric scale regularization
!python scripts/train_dino_ablation.py --ablation no-scale-loss --epochs 40 --batch-size 8 --accum-steps 4


In [ ]:
# [Ablation 6] Retrain WITHOUT Dynamic Pinhole Crop (Fixed Intrinsics K)
# Tests: Internalization of optical focal equivariance vs static positional memorization
!python scripts/train_dino_ablation.py --ablation no-dynamic-crop --epochs 40 --batch-size 8 --accum-steps 4


--- 
### Benchmark All Retrained Ablation Checkpoints on the 200 Unseen Frames
Once training completes, evaluate all checkpoints against the continuous held-out benchmark:


In [ ]:
# [Evaluation] Run Benchmark across all trained ablation checkpoints
!python scripts/download_and_eval_200.py

# Package checkpoints, logs, and plots for download
!zip -r -q dioptra_dino_retrained_ablations.zip outputs_ablations/
print("Ablation archive created: dioptra_dino_retrained_ablations.zip")
